# Trabajo Práctico: Calibración y Pose de la Cámara

**Universidad de Buenos Aires**  
Facultad de Ciencias Exactas y Naturales

**Materia:** Álgebra Lineal Computacional  
**Período:** Segundo cuatrimestre de 2026


## Grupo: milanesa SVD

### Integrantes

| Nombre y apellido | LU |
| --- | --- |
| Rocco Gasparini | 102/24 |
| Benjamin Pondal | 551/24 |
| Ivan Schuster | 401/24 |
| Lautaro Hildt | 1166/24 |

## Resolución


### Ejercicio 1 - DLT ecuaciones formales

**a)** Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

**1. Rango de la matriz $Q$:**

La matriz $Q$ tiene dimensiones $2n \times 12$ y el vector $m$ pertenece a $\mathbb{R}^{12}$. Por el Teorema de la Dimensión, sabemos que:
$$ \dim(\mathbb{R}^{12}) = \text{Rango}(Q) + \dim(\text{Nu}(Q)) $$
$$ 12 = \text{Rango}(Q) + \dim(\text{Nu}(Q)) $$

Como el enunciado exige una solución no nula y única a menos de un factor de escala, el núcleo de la matriz tiene que ser igual a $1$. Despejando, obtenemos que el rango necesario de la matriz es:
$$ \text{Rango}(Q) = 11 $$

**2. Cantidad mínima de puntos:**

Cada correspondencia de puntos aporta exactamente 2 ecuaciones al sistema (dos filas a la matriz $Q$). Para alcanzar un rango de 11, necesitamos como mínimo 11 filas linealmente independientes.

* Con 6 puntos obtenemos 12 filas, lo cual es suficiente para alcanzar el rango 11 requerido.

**3. Condiciones de los puntos:**
Para garantizar que el sistema alcance el rango 11, las coordenadas de los puntos elegidos no pueden ser arbitrarias.

* **Condición en el espacio 3D:**
Sin pérdida de generalidad, asumamos que todos los puntos de calibración pertenecen al plano $W = 0$. En la construcción de la matriz $Q$, las tres incógnitas de la cámara que multiplican a $W$ ($m_{13}, m_{23}$ y $m_{33}$) se multiplicarán por cero en todas las ecuaciones. Al no participar del sistema, estas 3 incógnitas quedan como variables libres, dejando a la matriz con un rango máximo de 8.

Para restringir estas 3 variables libres y lograr solución única, necesitamos aportar ecuaciones donde $W \neq 0$. Cada punto no coplanar aporta 2 ecuaciones. Si agregamos 1 solo punto fuera del plano, aportamos 2 ecuaciones, pero nos sigue quedando 1 variable libre (rango 10). Por lo tanto, es obligatorio agregar al menos 2 puntos fuera del plano base para lograr restringir el sistema. En conclusión, el conjunto total de puntos 3D no puede ser coplanar.

* **Condición en el plano de la imagen:**
Si todos los puntos proyectados en la imagen fueran colineales, podemos (sin pérdida de generalidad) rotar el sistema de referencia 2D para que caigan sobre el eje horizontal, donde $v_i = 0$ para todos los puntos. Al reemplazar $v = 0$, la ecuación impar ($f_{2i+1}$) se reduce a:
$$ U \cdot m_{21} + V \cdot m_{22} + W \cdot m_{23} + 1 \cdot m_{24} = 0 $$

Geométricamente, esta es la ecuación de un plano en el espacio 3D. 
Si todos nuestros puntos 3D satisfacen esta ecuación, significa que viven sobre ese único plano. Sin embargo, por la condición anterior ya demostramos que el conjunto de puntos 3D no puede ser coplanar. 
Por lo tanto, la única forma de que un conjunto no coplanar de puntos satisfaga esa ecuación simultáneamente es que los coeficientes del plano sean nulos: **$m_{21} = m_{22} = m_{23} = m_{24} = 0$**. 

Al forzar a cero estas 4 variables, nos quedan 8 incógnitas por despejar ($m_1$ y $m_3$). Como las ecuaciones impares ya se consumieron forzando los ceros, solo nos quedan las filas pares (las de $u$). Utilizando el mínimo de 6 puntos, disponemos de apenas 6 ecuaciones útiles para resolver 8 incógnitas. 
Este sistema subdeterminado garantiza la aparición de al menos 2 variables libres adicionales, provocando infinitas soluciones. Por ende, para que el algoritmo funcione, es estrictamente necesario que los puntos proyectados en la imagen no sean todos colineales.

**b)** Encontrar el vector $m$ que resuelve la minimización de la ecuación $8$.

El objetivo es encontrar el vector $m$ que minimice la forma cuadrática $m^T Q^T Q m$, sujeto a la restricción $\|m\| = 1$. Para resolverlo, utilizamos la Descomposición en Valores Singulares (SVD) de la matriz $Q = U \Sigma V^T$. Reemplazando $Q$ en la función a minimizar, obtenemos $m^T V (\Sigma^T \Sigma) V^T m$.

Para simplificar la expresión, definimos el vector $y = V^T m$ y calculamos su norma al cuadrado para verificar cómo se ve afectada la restricción inicial:
$$ \|y\|^2 = y^T y = (V^T m)^T (V^T m) = m^T V V^T m $$

Dado que $V$ es una matriz ortogonal proveniente de la SVD, se cumple que $V V^T = I$. Por lo tanto:
$$ \|y\|^2 = m^T I m = m^T m = \|m\|^2 $$

Como la norma inicial requerida era $\|m\| = 1$, queda demostrado que la norma del nuevo vector $y$ también es estrictamente 1.

Sabiendo que $y = V^T m$ y $y^T = m^T V$, el problema se transforma en minimizar la expresión $y^T (\Sigma^T \Sigma) y$ sujeta a la restricción $\|y\|^2 = 1$. La matriz $\Sigma^T \Sigma$ es diagonal y contiene los valores singulares al cuadrado ordenados de mayor a menor ($\sigma_1^2 \ge \sigma_2^2 \ge \dots \ge \sigma_{12}^2 \ge 0$). Al desarrollar este producto matricial, obtenemos una sumatoria:
$$ y^T (\Sigma^T \Sigma) y = \sum_{i=1}^{12} \sigma_i^2 y_i^2 $$

Como los valores singulares están ordenados, cualquier $\sigma_i^2$ es mayor o igual al último valor $\sigma_{12}^2$. Esto nos permite armar la siguiente desigualdad para acotar la función por debajo:
$$ \sum_{i=1}^{12} \sigma_i^2 y_i^2 \ge \sum_{i=1}^{12} \sigma_{12}^2 y_i^2 $$

Sacando factor común el término constante en el lado derecho:
$$ \sum_{i=1}^{12} \sigma_i^2 y_i^2 \ge \sigma_{12}^2 \left( \sum_{i=1}^{12} y_i^2 \right) $$

Por la restricción comprobada anteriormente, sabemos que la sumatoria de los $y_i^2$ es exactamente igual a la norma al cuadrado de $y$, que es 1. Sustituyendo esto, llegamos a la cota:
$$ \sum_{i=1}^{12} \sigma_i^2 y_i^2 \ge \sigma_{12}^2 $$

Para que un valor sea efectivamente el mínimo de una función, basta con verificar que la función es siempre mayor o igual a ese valor, y además demostrar que existe al menos un vector que hace posible alcanzar exactamente ese valor. Para que nuestra sumatoria sea igual a $\sigma_{12}^2$, basta con proponer un vector $y$ que asigne todo el peso a la última variable y anule las demás:
$$ y = \begin{bmatrix} 0 \\ 0 \\ \vdots \\ 1 \end{bmatrix} $$

Este vector propuesto cumple con la restricción de norma unitaria y hace que la función tome el valor exacto $\sigma_{12}^2$, demostrando de forma rigurosa que dicho valor es el mínimo buscado.

Finalmente, a partir de la definición $y = V^T m$, multiplicamos a izquierda por $V$ de ambos lados para despejar la variable original:
$$ V y = V V^T m \implies m = V y $$

Al multiplicar la matriz $V$ por el vector $y$ propuesto, el resultado extrae la última columna de la matriz. En conclusión, el vector $m$ de norma unitaria que resuelve la minimización original corresponde a la última columna de la matriz ortogonal $V$, la cual se obtiene al aplicar la SVD sobre la matriz de proyecciones $Q$.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



### Justificación de la implementación (Ejercicio 2b)

Para encontrar la matriz de proyección $M$, el problema se reduce a encontrar el vector $m$ que minimiza $\|Qm\|^2$ sujeto a $\|m\|=1$, lo cual equivale a encontrar la última columna de la matriz $V$ proveniente de la Descomposición en Valores Singulares (SVD) de la matriz $Q$. 

Por definición de la SVD, las columnas de la matriz ortogonal $V$ corresponden exactamente a los autovectores de la matriz simétrica $Q^T Q$. Por lo tanto, en lugar de computar la SVD completa o lidiar con los recortes por tolerancia de la SVD reducida, resulta más directo calcular los autovectores de $Q^T Q$. 

Para ello, construimos $Q^T Q$ utilizando la multiplicación matricial permitida por el módulo y le aplicamos la función diagRH. Dado que el método de la potencia combinado con reflectores de Householder extrae los autovalores ordenados de mayor a menor, el autovector asociado al menor autovalor se ubicará en la última columna de la matriz devuelta. Extraemos dicha columna y le aplicamos un cambio de forma geométrica a $3 \times 4$ para obtener la matriz de proyección final.

### Justificación de la implementación (Ejercicio 2c y 2d)

La matriz $M$ que obtenemos como resultado en el inciso 2.b surge de resolver un sistema homogéneo donde forzamos que la norma del vector $m$ sea igual a 1. Debido a esta restricción, la matriz calculada difiere de la matriz teórica por un factor de escala no nulo $p$. 

Por lo tanto, la relación entre nuestra matriz práctica y los parámetros físicos del modelo se plantea matricialmente como $p M = K [R \mid t]$. Escribiendo las matrices completas con todos sus componentes, la igualdad es la siguiente:

$$ 
p \begin{bmatrix} 
m_{11} & m_{12} & m_{13} & m_{14} \\ 
m_{21} & m_{22} & m_{23} & m_{24} \\ 
m_{31} & m_{32} & m_{33} & m_{34} 
\end{bmatrix} 
= 
\begin{bmatrix} 
\alpha_u & \gamma & u_0 \\ 
0 & \alpha_v & v_0 \\ 
0 & 0 & 1 
\end{bmatrix} 
\begin{bmatrix} 
r_{11} & r_{12} & r_{13} & t_1 \\ 
r_{21} & r_{22} & r_{23} & t_2 \\ 
r_{31} & r_{32} & r_{33} & t_3 
\end{bmatrix} 
$$

Notamos que al fijar el elemento $K_{33} = 1$, garantizamos algebraicamente que el factor de escala homogéneo $\lambda$ en la proyección coincida con la coordenada de profundidad $Z$ del punto en el sistema de la cámara, permitiendo a su vez determinar de forma unívoca la magnitud del factor de escala $p$.

Para facilitar el cálculo analítico, agrupamos los componentes espaciales en vectores:
* Definimos $q_i$ como el vector formado por los tres primeros elementos de la fila $i$ de la matriz $M$ (es decir, $q_1 = [m_{11}, m_{12}, m_{13}]$, etc.). 
* Definimos $r_i$ como el vector correspondiente a la fila $i$ de la matriz de rotación $R$ ($r_1 = [r_{11}, r_{12}, r_{13}]$, etc.). 
* Los elementos de la última columna de $M$ los llamamos $m_{14}, m_{24}, m_{34}$, y los del vector de traslación son $t_1, t_2, t_3$.

Al desarrollar la multiplicación matricial fila por fila y separar la submatriz de rotación del vector de traslación, obtenemos el siguiente sistema de ecuaciones:

$$ p q_1 = \alpha_u r_1 + \gamma r_2 + u_0 r_3 \tag{1} $$
$$ p q_2 = \alpha_v r_2 + v_0 r_3 \tag{2} $$
$$ p q_3 = r_3 \tag{3} $$
$$ p m_{14} = \alpha_u t_1 + \gamma t_2 + u_0 t_3 \tag{4} $$
$$ p m_{24} = \alpha_v t_2 + v_0 t_3 \tag{5} $$
$$ p m_{34} = t_3 \tag{6} $$

A su vez, por definición del modelo, sabemos que $R$ debe ser una matriz ortogonal ($R R^T = I$). Esto aporta un segundo bloque de ecuaciones fundamentales sobre sus filas: el producto interno entre dos filas distintas es cero, y la norma 2 de cada fila es igual a 1:

$$ r_1 \cdot r_2 = 0 \tag{7} $$
$$ r_1 \cdot r_3 = 0 \tag{8} $$
$$ r_2 \cdot r_3 = 0 \tag{9} $$
$$ \|r_1\|_2 = 1 \tag{10} $$
$$ \|r_2\|_2 = 1 \tag{11} $$
$$ \|r_3\|_2 = 1 \tag{12} $$

Con todas las ecuaciones planteadas, el proceso de despeje se realiza fila por fila:

1. **Tercera fila y magnitud de escala:**
   Tomamos la ecuación (3) y aplicamos la norma 2 a ambos lados. Por la ecuación (12), sabemos que $\|r_3\|_2 = 1$, por lo que obtenemos $\|p q_3\|_2 = |p| \|q_3\|_2 = 1$. Inicialmente tomamos el valor positivo del factor de escala:

   $$ p = \frac{1}{\|q_3\|_2} \tag{13} $$

   Con $p$ definido, la ecuación (3) nos da directamente el vector unitario $r_3 = p q_3$.

2. **Segunda fila ($v_0$, $\alpha_v$ y $r_2$):**
   Tomamos la ecuación (2) y realizamos el producto interno con $r_3$ a ambos lados: $p (q_2 \cdot r_3) = \alpha_v (r_2 \cdot r_3) + v_0 (r_3 \cdot r_3)$. Por las ecuaciones (9) y (12), se anula el término con $\alpha_v$ y despejamos $v_0$:

   $$ v_0 = p (q_2 \cdot r_3) \tag{14} $$

   Aislando el término de $\alpha_v$ en la ecuación (2), tenemos $\alpha_v r_2 = p q_2 - v_0 r_3$. Si reemplazamos $r_3 = p q_3$, podemos expresar el lado derecho como la combinación lineal $p q_2 - (v_0 p) q_3$. Para que la norma de este vector fuera nula ($\alpha_v = 0$), el vector debería ser el vector nulo ($p q_2 - v_0 p q_3 = 0$). Como las filas de la submatriz izquierda de $M$ ($q_1, q_2, q_3$) son linealmente independientes, la única forma de que su combinación lineal dé cero es que los escalares sean nulos (en particular, $p = 0$). Dado que en la ecuación (13) probamos que $p \neq 0$, llegamos a un absurdo. Esto garantiza matemáticamente que el vector es no nulo y, por lo tanto, su norma euclídea es estrictamente positiva:

   $$ \alpha_v = \|p q_2 - v_0 r_3\|_2 > 0 \tag{15} $$

   Dividiendo por $\alpha_v$, obtenemos el vector unitario $r_2 = \frac{p q_2 - v_0 r_3}{\alpha_v}$.

3. **Primera fila ($u_0$, $\gamma$, $\alpha_u$ y $r_1$):**
   Aplicamos el producto interno con $r_3$ a ambos lados de la ecuación (1). Por las ecuaciones (8), (9) y (12), desaparecen los términos de $\alpha_u$ y $\gamma$, despejando $u_0$:

   $$ u_0 = p (q_1 \cdot r_3) \tag{16} $$

   Aplicamos ahora el producto interno con $r_2$ en la ecuación (1). Por las ecuaciones (7), (9) y (11), se anulan los términos de $\alpha_u$ y $u_0$, despejando el sesgo (skew) $\gamma$:

   $$ \gamma = p (q_1 \cdot r_2) \tag{17} $$

   Agrupando los términos conocidos en la ecuación (1), obtenemos $\alpha_u r_1 = p q_1 - \gamma r_2 - u_0 r_3$. Al igual que en la fila anterior, al expandir $r_2$ y $r_3$ todos los términos se pueden reescribir como una combinación lineal de $q_1, q_2$ y $q_3$ donde el coeficiente que multiplica a $q_1$ es exactamente $p$. Por independencia lineal y siendo $p \neq 0$, el vector resultante jamás es nulo, lo que nos permite despejar $\alpha_u$ como su norma estrictamente positiva:

   $$ \alpha_u = \|p q_1 - \gamma r_2 - u_0 r_3\|_2 > 0 \tag{18} $$

   Dividiendo este vector resultante por $\alpha_u$, obtenemos el primer vector unitario $r_1 = \frac{p q_1 - \gamma r_2 - u_0 r_3}{\alpha_u}$.

4. **Condición de rotación propia ($\det(R) = +1$) y signo de $p$:**
   Por construcción, las filas $r_1, r_2, r_3$ son ortogonales entre sí y de norma unitaria, por lo que $R R^T = I$, lo que implica $\det(R) = \pm 1$. Sin embargo, una matriz ortogonal con $\det(R) = -1$ representa una reflexión y no una rotación. 
   Dado que en el inciso 2.b el vector $m$ se obtiene del problema homogéneo $\min_{\|m\|_2=1} \|Qm\|_2^2$, tanto $M$ como $-M$ son soluciones válidas, lo que equivale a que en la ecuación (13) el factor de escala pueda ser $p = \pm \frac{1}{\|q_3\|_2}$. 
   Si la matriz $R$ construida con $p > 0$ resulta con $\det(R) = -1$, invertimos el signo global tomando $p \leftarrow -p$ y $R \leftarrow -R$. Al ser $R \in \mathbb{R}^{3\times 3}$, se cumple que $\det(-R) = (-1)^3 \det(R) = +1$, garantizando que $R$ sea una rotación propia. Notamos que cambiar simultáneamente el signo de $p$ y de $r_1, r_2, r_3$ deja invariantes a $v_0, u_0, \gamma, \alpha_v$ y $\alpha_u$ en las ecuaciones (14) a (18), conservando $\alpha_u > 0$ y $\alpha_v > 0$.

5. **Restricciones opcionales sobre $K$ (Inciso 2d) y despeje de $t$:**
   La factorización de la submatriz izquierda $p M[:, :3] = K R$ en una matriz triangular superior con diagonal positiva y una matriz ortogonal es única. Cuando los datos contienen ruido de medición, si se imponen restricciones adicionales sobre $K$ ($\gamma = K_{12} = 0$ y/o $\alpha_u = \alpha_v$), no existe ninguna matriz estrictamente ortogonal que reconstruya de manera exacta la matriz ruidosa original. 
   Para preservar rigurosamente la condición de ortonormalidad $R R^T = I$ y $\det(R) = +1$, la matriz de rotación $R$ se mantiene intacta tal como fue ortonormalizada en los pasos previos, y las restricciones se imponen proyectando los parámetros intrínsecos de $K$:
   * Si se solicita ausencia de sesgo (`fuerza_k12_0 = True`), se fija $\gamma = 0$.
   * Si se solicita igualdad de escalas focales (`fuerza_au_av_iguales = True`), se adopta el valor promedio para ambos ejes: $\alpha_u = \alpha_v = \frac{\alpha_{u,\text{real}} + \alpha_{v,\text{real}}}{2}$.

   Finalmente, con el signo definitivo de $p$ y los parámetros finales de $K$, despejamos de abajo hacia arriba las componentes del vector de traslación $t = [t_1, t_2, t_3]^T$ a partir de las ecuaciones (6), (5) y (4):

   $$ t_3 = p m_{34}, \qquad t_2 = \frac{p m_{24} - v_0 t_3}{\alpha_v}, \qquad t_1 = \frac{p m_{14} - \gamma t_2 - u_0 t_3}{\alpha_u} \tag{19} $$

### Ejercicio 3 - Calibración con puntos coplanares IMPORTANTE QUE HAGO CON LA IMAGEN 20 QUE FALTA

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


#### Ítem 1 - Condición sobre el rango de $Q$

¿Qué rango debe tener $Q\in\mathbb{R}^{2n\times9}$ para asegurar una solución única de $H$ a menos de un factor de escala?

La matriz $Q\in\mathbb{R}^{2n\times 9}$ debe tener rango 8 para asegurar una solución única de $H$ a menos de un factor de escala.

El vector $h$ contiene los nueve elementos de la matriz $H$ y pertenece a $\mathbb{R}^{9}$. El conjunto de soluciones de la ecuación $Qh=0$ es el núcleo de $Q$. El vector $h=0$ siempre es solución. Sin embargo, produciría $H=0$, que no representa una homografía válida. Por eso buscamos una solución no nula.

Por otro lado, buscamos una solución única de $H$ a menos de un factor de escala. Esto quiere decir que, si $H_1$ y $H_2$ son dos soluciones no nulas, entonces $H_1=cH_2$ para algún escalar $c\neq0$. Multiplicar $H$ por un escalar no nulo no cambia las coordenadas de la imagen, porque ese factor se cancela al dividir por la tercera coordenada homogénea. Para que todas las soluciones no nulas sean múltiplos entre sí, el espacio de soluciones debe tener dimensión uno o, lo que es lo mismo, $\dim(\operatorname{Nu} Q)=1$.

Como $Q$ tiene nueve columnas, representa una transformación lineal con dominio $\mathbb{R}^{9}$. Por el teorema de la dimensión,

$$
\dim(\operatorname{Im} Q)+\dim(\operatorname{Nu} Q)=9.
$$

Recordemos que $\dim(\operatorname{Im} Q)=\operatorname{rg}(Q)$ y acabamos de ver que $\dim(\operatorname{Nu} Q)=1$. Luego,

$$
\operatorname{rg}(Q)+1=9,
$$

por lo tanto, $\operatorname{rg}(Q)=8$.


#### Ítem 2 - Construcción de $Q$

`obtenerQ2d(imgPts, objPts)` recibe las correspondencias entre puntos del tablero $(U_i,V_i)$ y de la imagen $(u_i,v_i)$, en el mismo orden. Cada correspondencia aporta las dos ecuaciones de la ecuación 11. Al apilarlas se obtiene $Q\in\mathbb{R}^{2n\times9}$ para el sistema $Qh=0$, donde $h$ contiene las entradas de $H$ ordenadas por filas.

Para obtener las homografías que se usan en el ítem siguiente, normalizamos los puntos según la sección 2.3, construimos $Q$ con esos puntos normalizados y elegimos el autovector del menor autovalor de $Q^TQ$, pidiendo que $\|h\|_2=1$. Por último, desnormalizamos $H$ para que vuelva a relacionar las coordenadas originales del tablero y de la imagen: $H=T_{\mathrm{img}}^{-1}\check H T_{\mathrm{obj}}$.


#### Ítem 3 - Construcción de $A$

`armarA(Hlst)` recibe una lista de homografías, cada una obtenida de una imagen distinta. Como la matriz de rotación es ortogonal entonces, en particular, las dos primeras columnas de la rotación son ortogonales entre si, y ambas tienen norma 1. Esto nos provee de dos restricciones por homografía: $c_1^TBc_2=0$ y $c_1^TBc_1-c_2^TBc_2=0$, con $B=K^{-T}K^{-1}$ y $c_1,c_2$ columnas de $H$.

Apilamos sus coeficientes como filas $v_{12}^T$ y $(v_{11}-v_{22})^T$, en el orden de incógnitas $b=(B_{11},B_{12},B_{22},B_{13},B_{23},B_{33})^T$. Así, obtenemos $Ab=0$, con $A\in\mathbb{R}^{2N\times6}$ para $N$ homografías.


#### Ítem 4 - Cálculo de la matriz intrínseca $K$

`calcularMatrizK2D(A)` obtiene $b$ como un autovector unitario del menor autovalor de $A^TA$, resolviendo $\min_{\|b\|_2=1}\|Ab\|_2^2$. Reconstruye la matriz simétrica $B=K^{-T}K^{-1}$ con el orden de incógnitas del ítem 3. Como el signo de $b$ es arbitrario, prueba $B$ y $-B$ y utiliza el que admite la factorización de Cholesky $B=LL^T$.

De esa relación se recupera $K=(L^T)^{-1}$ y se normaliza dividiendo por $K_{33}$. Así, $K$ es triangular superior, sus distancias focales son positivas y $K_{33}=1$. Si ninguno de los signos permite Cholesky, la función devuelve `None`.


#### Ítem 5 - Recuperación de la pose

Para una homografía $H=[c_1\ c_2\ c_3]$, la relación $H=sK[r_1\ r_2\ t]$ permite obtener columnas proporcionales a $r_1$, $r_2$ y $t$ mediante $K^{-1}H$. Se recupera la escala con $|\lambda|=1/\|K^{-1}c_1\|_2$ y se elige su signo para que el origen del tablero quede delante de la cámara.

`obtenerPose2D(Hj, K)` aplica esa escala, ortonormaliza las primeras dos columnas mediante Gram-Schmidt y construye $r_3=r_1\times r_2$. Así obtiene una rotación con $R^TR=I$ y $\det(R)=1$ y devuelve $(R,t)$ con la convención $p_{\mathrm{cámara}}=Rp_{\mathrm{tablero}}+t$.

Observar que la traslación tiene las unidades del tablero, no es directamente la posición de la cámara en el mundo.


#### Datos y correspondencias

Trabajamos con las 77 esquinas internas del tablero. Les asignamos coordenadas $(U,V)=(30i,30j)$ mm, con $i=0,\ldots,10$ y $j=0,\ldots,6$, recorriendo primero las columnas de cada fila. Elegimos como origen la esquina interna superior izquierda del tablero en cada imagen.

**ACLARACION Material disponible:** la carpeta contiene únicamente las imágenes 1 a 10. Usamos esas diez imágenes para la calibración. No disponemos de las imágenes 11 a 20 mencionadas en la plantilla, por lo que no podemos informar la pose de la imagen 20.


**Correspondencias pendientes:** los archivos recibidos no incluyen las coordenadas en píxeles de las esquinas del tablero coplanar. Falta confirmar con la cátedra si existe un archivo adicional o cómo se deben obtener. Por ese motivo, no presentamos todavía resultados numéricos de la calibración en esta versión.

Las celdas siguientes dejan preparado el cálculo una vez disponibles esas coordenadas. `puntos_imagen` debe asociar cada nombre de imagen con un arreglo de tamaño (77, 2), con las coordenadas en píxeles en el mismo orden que `objPts`.


In [ ]:
import numpy as np
from alc import obtenerQ2d, estimarHomografia, armarA, calcularMatrizK2D, obtenerPose2D

objPts = np.array([[30 * i, 30 * j] for j in range(7) for i in range(11)], dtype=float)

# Completar con las correspondencias cuando se confirme cómo obtenerlas.
puntos_imagen = {}
archivos = sorted(puntos_imagen, key=lambda nombre: int(nombre.split('_')[-1].split('.')[0]))

print('Puntos del tablero:', objPts.shape[0])
if not puntos_imagen:
    print('Pendiente: incorporar las coordenadas en píxeles de las esquinas.')


#### Homografías de las imágenes

Con las correspondencias anteriores calculamos una homografía por imagen mediante `estimarHomografia`, que realiza la normalización, el DLT y la desnormalización explicados en el ítem 2. Cada homografía relaciona las coordenadas del tablero en milímetros con los píxeles de su imagen.


In [ ]:
homografias = {}
for nombre in archivos:
    homografias[nombre] = estimarHomografia(puntos_imagen[nombre], objPts)

Hlst = [homografias[nombre] for nombre in archivos]
if Hlst:
    print('Homografías calculadas:', len(Hlst))
    print('Dimensiones de Q por imagen:', obtenerQ2d(puntos_imagen[archivos[0]], objPts).shape)
else:
    print('Homografías pendientes de las correspondencias en píxeles.')


#### a) Obtener la matriz $\mathbf{K}$

Las diez homografías aportan dos restricciones cada una. Construimos $A$ con `armarA` y recuperamos una única matriz intrínseca $K$, común a todas las imágenes, mediante `calcularMatrizK2D`.


In [ ]:
A = None
K = None
if Hlst:
    A = armarA(Hlst)
    K = calcularMatrizK2D(A)
    if K is None:
        raise ValueError('Las homografías no produjeron una matriz B definida positiva.')

    np.set_printoptions(precision=6, suppress=True)
    print('Dimensiones de A:', A.shape)
    print('K =')
    print(K)
    print('Distancias focales (píxeles):', K[0, 0], K[1, 1])
    print('Centro principal (píxeles):', K[0, 2], K[1, 2])
    print('Sesgo (píxeles):', K[0, 1])
else:
    print('K pendiente: primero se necesitan las homografías.')


La diagonal de K contiene las dos distancias focales en píxeles y el valor de normalización 1. Las entradas de la tercera columna indican el centro principal, mientras que la entrada K[0,1] es el sesgo. Estimamos estos parámetros sin imponer distancias focales iguales ni sesgo nulo.


#### b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$

Usamos la misma K y la homografía correspondiente a cada imagen. Mostramos R y t para las imágenes solicitadas que están disponibles.


In [ ]:
poses = {}
if K is None:
    print('Poses pendientes: primero se necesitan las homografías y K.')
else:
    for nombre in ['Im_R_1.png', 'Im_R_10.png', 'Im_R_20.png']:
        if nombre not in homografias:
            print(nombre + ': no se puede calcular la pose porque faltan sus correspondencias.')
            continue
        R, t = obtenerPose2D(homografias[nombre], K)
        poses[nombre] = (R, t)
        print(nombre)
        print('R =')
        print(R)
        print('t (mm) =', t)
        print()


Las poses se expresan respecto del sistema de coordenadas del tablero elegido en cada imagen: R describe su orientación respecto de la cámara y t ubica su origen en coordenadas de cámara, en milímetros. La calibración utiliza las diez imágenes disponibles. 

### la pose de la imagen 20 queda pendiente por falta de ese archivo.


### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---